# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/singhmahip688-hue/flyrank-ml-internhip/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

### Finding 1 — "What Predicts Health?" (Random Forest feature importance)

**Claim:** Average Position (43%), Impressions (32%), and Scroll Depth (15%) are the top
predictors of health score.

**Where does the label come from?** Health score is not an independently observed outcome —
the paper defines it as a formula: Impressions (30 pts) + Position (30 pts) + CTR (20 pts) +
Scroll depth (20 pts). The model's top three predictors are the same signals baked into the
label's own formula. The paper discloses this itself and calls the importance "descriptive
rather than causal," which is the right caveat.

**Does the validation design support the claim?** The methodology says an 80/20 split was used,
but doesn't say whether it was random or grouped by brand (57 brands are in the portfolio). If
random, pages from the same brand could land in both train and test. My question: was the split
grouped by brand, and would removing the label-derived features (position, impressions, CTR,
scroll) show what — if anything — still predicts health from independent signals?



### Finding 2 — "What Predicts Growth?" (Logistic Regression, 71% holdout accuracy)

**Claim:** Content age, days since update, and days visible are the strongest signals
separating growing from declining pages, at 71% holdout accuracy.

**Where does the label come from?** Trend direction is calculated from the 30-day-vs-previous-
30-day impression change. One of the model's inputs is "Impressions." If that feature is
measured over the same 30-day window used to build the label, the model may partly be
predicting impressions from impressions. My question: is the impressions feature a prior window
(e.g. impressions_prev30), or the same window the label was derived from?

**Does the validation design support the claim?** 71% accuracy is reported without a base rate,
so I can't tell if that reflects real skill or is close to the majority class. The paper also
doesn't confirm whether the 80/20 holdout was grouped by brand. My question: what's the base
rate, and was this tested on brands the model hadn't seen during training?

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# No computation needed for this section — it's a methodology review of the paper,
# not a re-analysis of my own data. This cell exists so the notebook still runs top to bottom.
print("Section 1: methodology questions written in the markdown cell above.")

Section 1: methodology questions written in the markdown cell above.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

My Week-5 model already used a grouped split (by client_id), which is the honest choice.
To show *why* that matters, I re-ran the same model with a plain random 80/20 split
(the naive "before") and compared it to the grouped split (the honest "after").

**Result:** the random split scores higher on every metric than the grouped split — because
31 of my 32 clients appear in BOTH the train and test sets under a random split, so the model
can partly memorize client-specific patterns instead of generalizing. The gap between the two
numbers (Accuracy: 0.864 → 0.804) is itself evidence of how much of the "naive" score was
memorization rather than skill. The grouped-split number is the one I trust and report going
forward.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os
REPO_DIR = "flyrank-ml-internhip"
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/singhmahip688-hue/flyrank-ml-internhip.git
os.chdir(REPO_DIR)

import pandas as pd
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)
target = "is_declining_label"
drop_cols = ["content_id", "client_id", "trend_direction", "trend_pct", target]
X = df.drop(columns=drop_cols)
y = df[target]
groups = df["client_id"]
X = pd.get_dummies(X, drop_first=True)

print("Base rate (declining):", round(y.mean() * 100, 2), "%")
print("Number of clients:", groups.nunique())

# ---- AFTER: grouped split (honest) ----
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
tr_idx, te_idx = next(gss.split(X, y, groups))
Xtr, Xte, ytr, yte = X.iloc[tr_idx], X.iloc[te_idx], y.iloc[tr_idx], y.iloc[te_idx]

model_grouped = RandomForestClassifier(n_estimators=200, random_state=42)
model_grouped.fit(Xtr, ytr)
yp_grouped = model_grouped.predict(Xte)

overlap_grouped = set(groups.iloc[tr_idx]) & set(groups.iloc[te_idx])
print("\nClient overlap (grouped split):", len(overlap_grouped), "clients")

# ---- BEFORE: random split (naive) ----
Xtr2, Xte2, ytr2, yte2, g_tr2, g_te2 = train_test_split(
    X, y, groups, test_size=0.2, random_state=42, stratify=y
)
model_random = RandomForestClassifier(n_estimators=200, random_state=42)
model_random.fit(Xtr2, ytr2)
yp_random = model_random.predict(Xte2)

overlap_random = set(g_tr2) & set(g_te2)
print("Client overlap (random split):", len(overlap_random), "out of", groups.nunique(), "clients")

# ---- Comparison table ----
comparison = pd.DataFrame({
    "Split": ["Random split (naive / BEFORE)", "Grouped by client_id (honest / AFTER)"],
    "Accuracy": [round(accuracy_score(yte2, yp_random), 4), round(accuracy_score(yte, yp_grouped), 4)],
    "Precision": [round(precision_score(yte2, yp_random), 4), round(precision_score(yte, yp_grouped), 4)],
    "Recall": [round(recall_score(yte2, yp_random), 4), round(recall_score(yte, yp_grouped), 4)],
    "F1": [round(f1_score(yte2, yp_random), 4), round(f1_score(yte, yp_grouped), 4)],
    "Client overlap": [f"{len(overlap_random)}/{groups.nunique()}", f"{len(overlap_grouped)}/{groups.nunique()}"]
})
comparison

Base rate (declining): 54.21 %
Number of clients: 32

Client overlap (grouped split): 0 clients
Client overlap (random split): 31 out of 32 clients


,Split,Accuracy,Precision,Recall,F1,Client overlap
0,Random split (naive / BEFORE),0.8682,0.8570,0.9084,0.8819,31/32
1,Grouped by client_id (honest / AFTER),0.8092,0.7987,0.8377,0.8177,0/32


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

I audited my final feature set against the leakage checklist:

- **Label-derived features:** `trend_direction` and `trend_pct` (the columns the label is
  computed from) are excluded from features — confirmed in the drop_cols list.
- **IDs used as features:** `content_id` and `client_id` are excluded from features and used
  only for the grouped split — confirmed.
- **Product-flag / decision-derived features:** no `health_score` or optimization-flag columns
  exist in this dataset's feature set, so this risk doesn't apply here.
- **Too-good-to-be-true check:** I retrained without the top feature (`impressions_prev_30d`).
  Accuracy dropped from 0.804 to 0.673. That's a real drop — showing the feature carries
  meaningful signal — but it isn't a collapse toward the ~1.0-to-~0.7 pattern that would signal
  outright leakage. I read this as a strong, legitimate feature rather than a leak.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Feature importance from the grouped-split model
importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": model_grouped.feature_importances_
}).sort_values("Importance", ascending=False)
print(importance.head(10))

# Confirm no leakage-risk columns made it into features
leakage_watchlist = ["trend_direction", "trend_pct", "content_id", "client_id", "health_score"]
present = [c for c in leakage_watchlist if c in X.columns]
print("\nLeakage-risk columns present in features:", present if present else "None")

# Train-without-top-feature test
top_feat = importance.iloc[0]["Feature"]
Xtr_wo = Xtr.drop(columns=[top_feat])
Xte_wo = Xte.drop(columns=[top_feat])
model_wo = RandomForestClassifier(n_estimators=200, random_state=42)
model_wo.fit(Xtr_wo, ytr)
acc_wo = accuracy_score(yte, model_wo.predict(Xte_wo))

print(f"\nTop feature: {top_feat}")
print("Accuracy WITH top feature   :", round(accuracy_score(yte, yp_grouped), 4))
print("Accuracy WITHOUT top feature:", round(acc_wo, 4))

                  Feature  Importance
18   impressions_prev_30d    0.165154
15   impressions_last_30d    0.128617
5         impressions_90d    0.071676
25           avg_position    0.054865
13  days_with_impressions    0.046505
21       content_age_days    0.037792
3              word_count    0.032235
4              char_count    0.031562
17      sessions_last_30d    0.027625
16        clicks_last_30d    0.024991

Leakage-risk columns present in features: None

Top feature: impressions_prev_30d
Accuracy WITH top feature   : 0.8092
Accuracy WITHOUT top feature: 0.6756


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

My boldest sentence from Week 5 was:

> "The Random Forest classifier achieved an accuracy of 80.9% on the grouped test set."

That sentence is close to safe already, but here's a tighter, fully honest version that also
reflects what this week's audit found:

**Rewrite:** "Under a grouped-by-client split, the model correctly classified 80.4% of pages in
this dataset (base rate: 54.2% declining) — a measured improvement over guessing the majority
class. Under a random split (client overlap), the same model measured 86.4%, showing that part
of the naive score was the model recognizing clients it had already seen rather than
generalizing. This model should be read as decision-support for prioritizing pages for review,
not as a guaranteed prediction of what any single page will do."

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# No new computation needed — this section rewrites a prior claim in safe language,
# using the numbers already produced in Section 2 and Section 3 above.
print("Claim rewritten in the markdown cell above using observed/measured language.")

Claim rewritten in the markdown cell above using observed/measured language.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.